# Instructor 101 — Typed LLM Outputs in 5 Minutes

**Week 1 | Notebook 1 of 4**

**What you'll learn:**
- One-line client patching with Instructor
- Extracting typed Pydantic objects from any LLM
- Field descriptions as prompts
- Nested models and optional fields
- Type safety with IDE autocomplete

**Runtime:** ~25 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("01_instructor/01_core_extraction.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  01_instructor/01_core_extraction.ipynb
Task:      Basic extraction with Instructor
Calls:     ~10

With GPT-4o:       $0.10 USD
With GPT-4o-mini:  $0.01 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup & Client Patching

In [2]:
from pydantic import BaseModel, Field

from src.config import USE_OLLAMA, get_instructor_client, get_openai_client, print_config

print_config()

LLM Libraries — Configuration
USE_OLLAMA:        False
USE_SMALL_MODEL:   False
OLLAMA_BASE_URL:   http://localhost:11434
OpenAI model:      gpt-4o
Anthropic model:   claude-opus-4-6
SAMPLE_SIZE:       50
DSPY_TRIALS:       10


In [3]:
# Patch the client with Instructor — ONE LINE
# Uses the provider set by LLM_PROVIDER in .env (default: openai)
client = get_instructor_client()

# Override per-call if you like — gemini and groq have free API tiers
# client = get_instructor_client("gemini")
# client = get_instructor_client("groq")
# client = get_instructor_client("anthropic")

## 2. Your First response_model — Simple User Extraction

In [4]:
class User(BaseModel):
    name: str = Field(description="The person's full name")
    age: int = Field(description="Age in years")
    email: str = Field(description="Valid email address")


user = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=User,
    messages=[
        {"role": "user", "content": "John Smith is 30 years old and his email is john@example.com"}
    ],
)

print(user)
print(type(user))  # <class '__main__.User'>
print(user.name)  # IDE autocomplete works!

name='John Smith' age=30 email='john@example.com'
<class '__main__.User'>
John Smith


## 3. Field Descriptions as Prompts

In [7]:
class ProductReview(BaseModel):
    """Extract structured data from a product review."""

    product_name: str = Field(description="Name of the product being reviewed")
    rating: int = Field(ge=1, le=5, description="Star rating from 1 to 5")
    pros: list[str] = Field(description="List of positive aspects mentioned")
    cons: list[str] = Field(description="List of negative aspects mentioned")
    would_recommend: bool = Field(description="Whether the reviewer would recommend this product")


review_text = """
The Sony WH-1000XM5 headphones are absolutely amazing. The noise cancellation is best-in-class
and the battery lasts forever. They're a bit expensive though, and the carrying case is bulky.
Overall I'd definitely recommend them to anyone who travels a lot.
"""

review = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=ProductReview,
    messages=[{"role": "user", "content": f"Extract review data from: {review_text}"}],
)

print(review.model_dump_json(indent=2))

{
  "product_name": "Sony WH-1000XM5",
  "rating": 5,
  "pros": [
    "best-in-class noise cancellation",
    "long battery life"
  ],
  "cons": [
    "a bit expensive",
    "bulky carrying case"
  ],
  "would_recommend": true
}


## 4. Nested Models — Address inside User

In [8]:
class Address(BaseModel):
    street: str
    city: str
    state: str
    zipcode: str = Field(pattern=r"^\d{5}(-\d{4})?$")


class Customer(BaseModel):
    name: str
    email: str
    shipping_address: Address
    billing_address: Address | None = None


customer_text = """
Customer: Jane Doe, jane@company.com
Shipping: 123 Main St, San Francisco, CA 94105
"""

customer = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=Customer,
    messages=[{"role": "user", "content": customer_text}],
)

print(f"Customer: {customer.name}")
print(f"City: {customer.shipping_address.city}")
print(f"Billing same as shipping: {customer.billing_address is None}")

Customer: Jane Doe
City: San Francisco
Billing same as shipping: True


## 5. Optional Fields — Handling Missing Data

In [9]:
class Contact(BaseModel):
    name: str
    email: str | None = None
    phone: str | None = None
    company: str | None = None


contacts = [
    "Alice Johnson, alice@example.com, works at Google",
    "Bob Smith, phone: 555-0199",
    "Charlie Brown",
]

for text in contacts:
    contact = client.chat.completions.create(
        model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
        response_model=Contact,
        messages=[{"role": "user", "content": text}],
    )
    print(
        f"{contact.name}: email={contact.email}, phone={contact.phone}, company={contact.company}"
    )

Alice Johnson: email=alice@example.com, phone=None, company=Google
Bob Smith: email=None, phone=555-0199, company=None
Charlie Brown: email=None, phone=None, company=None


## 6. Type Safety Demo — IDE Autocomplete, mypy Check

In [10]:
# Demonstrate that the output is fully typed
def process_customer(customer: Customer) -> str:
    """This function has full type safety thanks to Pydantic."""
    return f"{customer.name} lives in {customer.shipping_address.city}"


# mypy would catch this at type-check time:
# process_customer("not a customer")  # ERROR: Argument 1 has incompatible type "str"

print(process_customer(customer))

Jane Doe lives in San Francisco


## 7. Comparing: Raw JSON Prompting vs Instructor (Failure Rate)

In [11]:
import json

raw_client = get_openai_client()

# Raw JSON prompting (no Instructor)
response = raw_client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    messages=[
        {"role": "system", "content": "Return ONLY valid JSON with keys: name, age, email"},
        {"role": "user", "content": "Extract: Jane Doe, 28, jane@example.com"},
    ],
)

raw_text = response.choices[0].message.content
print("Raw output:")
print(raw_text)

# Try to parse — may fail if model adds markdown or extra text
try:
    parsed = json.loads(raw_text)
    print("\n✅ Parsed successfully")
except json.JSONDecodeError as e:
    print(f"\n❌ Parse failed: {e}")

# With Instructor — never fails, always typed
user = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=User,
    messages=[{"role": "user", "content": "Extract: Jane Doe, 28, jane@example.com"}],
)
print(f"\nInstructor output: {user}")

Raw output:
{
  "name": "Jane Doe",
  "age": 28,
  "email": "jane@example.com"
}

✅ Parsed successfully

Instructor output: name='Jane Doe' age=28 email='jane@example.com'
